### Load Data Hasil Preprocessing

Memastikan dataset hasil preprocessing (isi_berita_clean) sudah siap digunakan.

In [14]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

In [15]:
# Load dataset hasil preprocessing jika belum ada di memory
if 'df' not in locals():
    df = pd.read_csv('detik_preprocessed.csv')

# Tampilkan baris untuk verifikasi
print("=== VERIFIKASI DATA SIAP TF-IDF ===")
display(df[['id', 'isi_berita_clean', 'label']].head(200))

=== VERIFIKASI DATA SIAP TF-IDF ===


,id,isi_berita_clean,label
0,1,ketua umum komite olahraga nasional indonesia ...,sport
1,2,urus pusat kickboxing indonesia pp kbi periode...,sport
2,3,lina hisage atlet tolak peluru asal wamena pap...,sport
3,4,timnas equestrian indonesia tolak jepang jelan...,sport
4,5,dua seri akhir motogp marc marquez sukses sapu...,sport
...,...,...,...
195,196,purbaya yudhi sadewa laku proses serah terima ...,finance
196,197,pt bayan resources tbk byan umum ada kahar ali...,finance
197,198,bp bumn danantara dorong selesai soal lrt city...,finance
198,199,direktur jenderal pajak djp bimo wijayanto pas...,finance


### Inisialisasi & Fitting TfidfVectorizer

Mengubah teks berita (isi_berita_clean) menjadi matriks angka berbobot TF-IDF menggunakan scikit-learn.

In [16]:
# 1. Pemetaan Label ke Angka (sport -> 1, finance -> 0)
df['label_encoded'] = df['label'].map({'sport': 1, 'finance': 0})

# 2. Inisialisasi & Fit-Transform TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['isi_berita_clean'])

print("=== HASIL ENCODING LABEL & FIT TF-IDF ===")
print("• Distribusi Label Numerical:")
print(df['label_encoded'].value_counts().to_string())
print(f"\n• Dimensi Matriks TF-IDF : {tfidf_matrix.shape} (Artikel x Kata Unik)")

=== HASIL ENCODING LABEL & FIT TF-IDF ===
• Distribusi Label Numerical:
label_encoded
1    100
0    100

• Dimensi Matriks TF-IDF : (200, 5307) (Artikel x Kata Unik)


### Ekstraksi Vocabulary (Daftar Kata Unik)

Mengambil semua kata unik (features) yang berhasil diekstrak oleh vectorizer.

In [21]:
# Ambil daftar seluruh nama fitur (kata unik)
feature_names = tfidf_vectorizer.get_feature_names_out()

print("=== HASIL EKSTRAKSI VOCABULARY ===")
print(f"• Total Kata Unik    : {len(feature_names)} kata/fitur\n")

print("📌 Sampel 15 Kata Unik Pertama:")
print(list(feature_names[:20]))

=== HASIL EKSTRAKSI VOCABULARY ===
• Total Kata Unik    : 5307 kata/fitur

📌 Sampel 15 Kata Unik Pertama:
['aaa', 'aau', 'ab', 'abad', 'abadi', 'abai', 'abal', 'abdullah', 'abdurrahkman', 'abet', 'abha', 'abraham', 'abrahham', 'absen', 'absorber', 'abu', 'academy', 'acara', 'accelerating', 'acceptance']


### Pembentukan DataFrame Matriks TF-IDF

Mengonversi matriks terkompresi (sparse matrix) menjadi tabel DataFrame Pandas agar bisa dibaca dengan mudah, serta menyisipkan kolom id dan label.

In [24]:
# 1. Ambil daftar seluruh kata unik (features)
feature_names = tfidf_vectorizer.get_feature_names_out()

# 2. Ubah matriks TF-IDF ke DataFrame Pandas
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)

# 3. Sisipkan kolom 'id' dan 'label' (1 / 0) di posisi awal
# Hapus metadata jika sel dijalankan ulang agar tidak terjadi duplikasi kolom
df_tfidf = df_tfidf.drop(columns=['id', 'label'], errors='ignore')
df_tfidf.insert(0, 'label', df['label_encoded'].to_numpy())
df_tfidf.insert(0, 'id', df['id'].to_numpy())

print("=== TABEL MATRIKS TF-IDF (LABEL 1 & 0) ===")
# Tampilkan 5 berita pertama dan 10 kata unik pertama
display(df_tfidf.iloc[:5, :12])

=== TABEL MATRIKS TF-IDF (LABEL 1 & 0) ===


,id,label,aaa,aau,ab,abad,abadi,abai,abal,abdullah,abdurrahkman,abet
0,1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Analisis Bobot Kata Kunci Tertinggi Per Kategori

Memeriksa kata-kata yang memiliki bobot TF-IDF paling dominan pada sampel berita kategori Sport dan Finance untuk menguji relevansi fitur.

In [25]:
# Ambil 1 sampel Sport (label == 1) dan 1 sampel Finance (label == 0)
sample_sport = df_tfidf[df_tfidf['label'] == 1].iloc[0]
sample_finance = df_tfidf[df_tfidf['label'] == 0].iloc[0]

def get_top_tfidf_words(series, top_n=5):
    words = series.drop(['id', 'label'])
    return words.sort_values(ascending=False).head(top_n)

print("=== TOP 5 KATA BERBOBOT TERTINGGI (SPORT - Label 1) ===")
print(f"ID Artikel: {int(sample_sport['id'])} | Label: {int(sample_sport['label'])}")
for word, score in get_top_tfidf_words(sample_sport).items():
    print(f"• {word:<15} : {score:.4f}")

print("\n=== TOP 5 KATA BERBOBOT TERTINGGI (FINANCE - Label 0) ===")
print(f"ID Artikel: {int(sample_finance['id'])} | Label: {int(sample_finance['label'])}")
for word, score in get_top_tfidf_words(sample_finance).items():
    print(f"• {word:<15} : {score:.4f}")

=== TOP 5 KATA BERBOBOT TERTINGGI (SPORT - Label 1) ===
ID Artikel: 1 | Label: 1
• timor           : 0.5270
• leste           : 0.4684
• marciano        : 0.2566
• koni            : 0.2449
• saudara         : 0.2449

=== TOP 5 KATA BERBOBOT TERTINGGI (FINANCE - Label 0) ===
ID Artikel: 101 | Label: 0
• stasiun         : 0.4945
• pramono         : 0.3141
• jakarta         : 0.2781
• lrt             : 0.2308
• prabowo         : 0.1875


### Simpan Matriks TF-IDF ke CSV

In [26]:
# Simpan matriks TF-IDF lengkap ke CSV
df_tfidf.to_csv('detik_tfidf_matrix.csv', index=False, encoding='utf-8-sig')

print("✅ Matriks TF-IDF dengan encoding label (1/0) berhasil dibuat!")
print("💾 File tersimpan sebagai 'detik_tfidf_matrix.csv'")

✅ Matriks TF-IDF dengan encoding label (1/0) berhasil dibuat!
💾 File tersimpan sebagai 'detik_tfidf_matrix.csv'
